In [1]:
import pandas as pd
import numpy as np
from ml_utils.src import *
from ml_utils.config import *

import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 1.0 - Classificação do Desempenho Baseado em Fatores Socioeconômicos Usando Random Forest

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet(DATA_DIR, columns = colunas)

## 1.1 - Pré-Processamento dos Dados 

In [3]:
df = preparar_dados_forests(df, objetivo = 'Desempenho', n_samples = 500_000)

## 1.2 - Construção da Matriz X e Vetor y

In [4]:
X = df.drop(columns=['MEDIA', 'FALTOU'])

y_media = df['MEDIA']

## 1.3 - Separação em Dados de Treino e Teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y_media, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [6]:
quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_val   = (y_val   >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

## 1.4 - Treinando o modelo

In [7]:
rf = RandomForestClassifier()

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eval: %0.4f' % (1 - accuracy_score(y_val,  rf.predict(X_val))))

print(classification_report(y_val, rf.predict(X_val)))

Ein:  0.0319
Eval: 0.3177
              precision    recall  f1-score   support

           0       0.68      0.69      0.69     25087
           1       0.68      0.67      0.68     24804

    accuracy                           0.68     49891
   macro avg       0.68      0.68      0.68     49891
weighted avg       0.68      0.68      0.68     49891



## 1.5 Treinando com os Melhores Parâmetros

In [8]:
cv_rf = buscar_hiperparametros_rf(X_train, y_train, n_iter=10, cv=5, scoring='accuracy', random_state=42)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, cv_rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_val,  cv_rf.predict(X_val))))

print(classification_report(y_val, cv_rf.predict(X_val)))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Ein:  0.2872
Eout: 0.2913
              precision    recall  f1-score   support

           0       0.70      0.74      0.72     25087
           1       0.72      0.68      0.70     24804

    accuracy                           0.71     49891
   macro avg       0.71      0.71      0.71     49891
weighted avg       0.71      0.71      0.71     49891



In [9]:
print(cv_rf.best_estimator_)

RandomForestClassifier(class_weight='balanced', max_depth=20, max_samples=0.8,
                       min_samples_leaf=50, min_samples_split=20,
                       n_estimators=300, random_state=42)


## 1.6 - Treinando com todos os dados de treino

In [11]:
rf = RandomForestClassifier(**cv_rf.best_params_, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y_media, test_size=0.2, random_state=42)

quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

#joblib.dump(rf, 'models/rf_desempenho.pkl')

Ein:  0.2863
Eout: 0.2958
              precision    recall  f1-score   support

           0       0.69      0.73      0.71     31206
           1       0.72      0.68      0.70     31158

    accuracy                           0.70     62364
   macro avg       0.70      0.70      0.70     62364
weighted avg       0.70      0.70      0.70     62364



In [8]:
rf = RandomForestClassifier(class_weight= None, max_depth=20, max_samples=0.8,
                       min_samples_leaf=50, min_samples_split=20,
                       n_estimators=300, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y_media, test_size=0.2, random_state=42)

quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

rf.fit(X_train, y_train)

print('Ein:  %0.4f' % (1 - accuracy_score(y_train, rf.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test,  rf.predict(X_test))))

print(classification_report(y_test, rf.predict(X_test)))

joblib.dump(rf, 'rf_desempenho.joblib')

Ein:  0.2908
Eout: 0.3005
              precision    recall  f1-score   support

           0       0.69      0.73      0.71     31206
           1       0.71      0.67      0.69     31158

    accuracy                           0.70     62364
   macro avg       0.70      0.70      0.70     62364
weighted avg       0.70      0.70      0.70     62364



['rf_desempenho.joblib']